# Fine-tune MultiVSR Encoder-Decoder on VSRo-200

Continues training the official MultiVSR encoder-decoder
(Prajwal et al., 2025) on the VSRo-200 face crops. The visual encoder
(VTP) is kept frozen; only the encoder-decoder is updated.

**Expected data layout** (after running the `dataset/` pipeline +
MultiVSR's preprocessing, then flattening the per-clip output):

```
<DATA_ROOT>/
├── ZWNM2sZxtRg/
│   ├── 00001.avi
│   ├── 00002.avi
│   └── ...
└── abc123def/
    ├── 00005.avi
    └── ...
```

The training/validation CSVs use `file_path,transcript` where
`file_path` is `<youtube_id>/<clip_index>` (no extension), matching
the format published as VSRo-200 splits.

In [ ]:
# Setup — clone MultiVSR (for the model code) and install deps
!git clone https://github.com/Sindhu-Hegde/multivsr.git
%cd multivsr
!pip install -q -r requirements.txt
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q huggingface_hub jiwer tqdm pandas numpy decord

# Download the pretrained MultiVSR checkpoints
!mkdir -p checkpoints
!wget -q -O checkpoints/feature_extractor.pth \
    "https://www.robots.ox.ac.uk/~vgg/research/vtp-for-lip-reading/checkpoints/extended_train_data/feature_extractor.pth"
!wget -q -O checkpoints/model.pth \
    "https://www.robots.ox.ac.uk/~vgg/research/multivsr/model.pth"

In [ ]:
# Imports
import sys
import os
import gc

sys.path.append("/multivsr")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import numpy as np
import pandas as pd
from tqdm import tqdm

from models import build_model, build_visual_encoder
from tokenizer import get_tokenizer
from config import pad_token

tokenizer = get_tokenizer()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# Configuration

# Path to your processed face crops (one .avi per clip, organized as
# DATA_ROOT/<youtube_id>/<clip_index>.avi)
DATA_ROOT = "/processed"

# Train/val CSVs (file_path, transcript), where file_path is
# <youtube_id>/<clip_index> (no extension)
TRAIN_CSV = "/train.csv"
VAL_CSV = "/val.csv"

# Where to save checkpoints (locally)
OUTPUT_DIR = "./checkpoints_finetuned"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Training hyperparameters
BATCH_SIZE = 6
ACCUMULATION_STEPS = 5
INITIAL_LR = 1e-4
PATIENCE_STOP = 4
SEED = 42

# Reproducibility
g = torch.Generator()
g.manual_seed(SEED)

print(f"Data root: {DATA_ROOT}")
print(f"Train CSV: {TRAIN_CSV}")
print(f"Val CSV:   {VAL_CSV}")
print(f"Output:    {OUTPUT_DIR}")

In [ ]:
# PAD ID + collate function
if hasattr(tokenizer, 'pad_token_id'):
    PAD_ID = tokenizer.pad_token_id
elif hasattr(tokenizer, 'vocab') and pad_token in tokenizer.vocab:
    PAD_ID = tokenizer.vocab[pad_token]
else:
    PAD_ID = 0
print(f"PAD_ID = {PAD_ID}")


def seq2seq_collate_fn(batch):
    batch = [item for item in batch if item is not None and item[0] is not None]
    if len(batch) == 0:
        return None, None
    videos, texts = zip(*batch)
    videos_transposed = [v.permute(1, 0, 2, 3) for v in videos]
    videos_padded = pad_sequence(videos_transposed, batch_first=True, padding_value=0)
    videos_final = videos_padded.permute(0, 2, 1, 3, 4)
    texts_padded = pad_sequence(texts, batch_first=True, padding_value=PAD_ID)
    return videos_final, texts_padded

In [ ]:
# Dataset: reads .avi files and converts them to tensors on-the-fly

class AVIDataset(Dataset):
    """
    Reads .avi face crops from DATA_ROOT/<youtube_id>/<clip_index>.avi
    given a CSV with columns (file_path, transcript), where file_path
    is "<youtube_id>/<clip_index>".
    """

    def __init__(self, data_root, csv_path, tokenizer):
        self.data_root = data_root
        self.tokenizer = tokenizer

        print(f"Loading split: {csv_path}")
        self.df = pd.read_csv(csv_path)
        self.df = self.df.dropna(subset=['transcript'])
        self.data = self.df.to_dict('records')
        print(f"  -> {len(self.data)} clips")

    def _file_path_to_avi(self, file_path):
        """`<youtube_id>/<clip_index>` -> path to its .avi"""
        return os.path.join(self.data_root, file_path + ".avi")

    def _load_avi(self, avi_path):
        """Convert an AVI to a tensor (C, T, H, W) of float32 in [0, 1].
        Returns None on error."""
        from decord import VideoReader, bridge
        bridge.set_bridge('native')

        try:
            with open(avi_path, 'rb') as f:
                vr = VideoReader(f, width=160, height=160)
                frames = vr.get_batch(list(range(len(vr)))).asnumpy()

            # Same processing as the original MultiVSR pipeline:
            #   normalize -> center crop 96x96 -> transpose to (C, T, H, W)
            frames = frames.astype(np.float32) / 255.0
            frames_crop = frames[:, 32:128, 32:128, :]              # (T, 96, 96, 3)
            video_array = np.transpose(frames_crop, (3, 0, 1, 2))    # (C, T, H, W)
            return torch.FloatTensor(video_array)
        except Exception as e:
            print(f"Warning: failed to load AVI {avi_path}: {e}")
            return None

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        fname = str(item['file_path'])
        avi_path = self._file_path_to_avi(fname)

        video = self._load_avi(avi_path)
        if video is None:
            return None, None

        text = str(item['transcript'])
        token_ids = self.tokenizer.encode(text)
        labels = torch.LongTensor(token_ids)

        return video, labels

In [ ]:
# Collate functions

def train_collate_fn(batch):
    batch = [item for item in batch if item is not None and item[0] is not None]
    if len(batch) == 0:
        return None, None
    videos, labels = zip(*batch)
    videos_transposed = [v.permute(1, 0, 2, 3) for v in videos]
    videos_padded = pad_sequence(videos_transposed, batch_first=True, padding_value=0)
    videos_final = videos_padded.permute(0, 2, 1, 3, 4)
    labels_padded = pad_sequence(labels, batch_first=True, padding_value=0)
    return videos_final, labels_padded


def test_collate_fn(batch):
    batch = [item for item in batch if item is not None and item[0] is not None]
    if len(batch) == 0:
        return None, None, None
    videos, texts, filenames = zip(*batch)
    videos_transposed = [v.permute(1, 0, 2, 3) for v in videos]
    videos_padded = pad_sequence(videos_transposed, batch_first=True, padding_value=0)
    videos_final = videos_padded.permute(0, 2, 1, 3, 4)
    return videos_final, texts, filenames

In [ ]:
# Load encoder-decoder + visual encoder

def load_model_and_visual(device="cuda"):
    """Load the encoder-decoder and the visual encoder (VTP, frozen)."""
    # Encoder-decoder (start from MultiVSR's original checkpoint)
    ckpt_path = "/multivsr/checkpoints/model.pth"
    model = build_model().to(device)
    checkpoint = torch.load(ckpt_path, map_location=device)
    s = checkpoint["state_dict"]
    new_s = {k[7:] if k.startswith("module.") else k: v for k, v in s.items()}
    model.load_state_dict(new_s)
    print(f"Encoder-decoder loaded: {ckpt_path}")

    # Visual encoder (VTP original, frozen)
    visual_encoder = build_visual_encoder().to(device)
    visual_ckpt_path = "/multivsr/checkpoints/feature_extractor.pth"
    s = torch.load(visual_ckpt_path, map_location=device)["state_dict"]
    new_s = {}
    for k, v in s.items():
        if "face_encoder" not in k:
            continue
        k_clean = k.replace("module.face_encoder.", "")
        new_s[k_clean] = v
    visual_encoder.load_state_dict(new_s)
    print(f"Visual encoder loaded: {visual_ckpt_path}")

    # Freeze visual encoder
    for p in visual_encoder.parameters():
        p.requires_grad = False
    visual_encoder.eval()
    model.train()

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters (encoder-decoder): {n_trainable:,}")

    return model, visual_encoder

In [ ]:
# Causal mask + validation function

def subsequent_mask(size):
    attn_shape = (1, size, size)
    mask = np.triu(np.ones(attn_shape), k=1).astype('uint8')
    return torch.from_numpy(mask) == 0


def validate(vis_encoder, lm, loader, criterion):
    lm.eval()
    vis_encoder.eval()
    total_loss = 0.0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Validating"):
            videos, full_targets = batch
            if videos is None:
                continue
            videos = videos.to(device)
            full_targets = full_targets.to(device)

            dec_in = full_targets[:, :-1]
            target = full_targets[:, 1:]
            L = dec_in.size(1)
            tgt_mask = (subsequent_mask(L).to(device) & (dec_in != PAD_ID).unsqueeze(-2))

            with torch.amp.autocast('cuda'):
                vid_emb = vis_encoder(videos)
                enc_B, enc_T, enc_D = vid_emb.size()
                src_mask = torch.ones((enc_B, 1, enc_T), device=device).bool()
                mem, _ = lm.encode(vid_emb, src_mask)
                logits = lm.decode(mem, src_mask, dec_in, tgt_mask)
                flat_targets = target.reshape(-1)
                loss = criterion(logits, flat_targets)

            total_loss += loss.item()

    return total_loss / max(len(loader), 1)

In [ ]:
# Initialize models, datasets, and DataLoaders

model, visual_encoder = load_model_and_visual(device=device)

train_dataset = AVIDataset(DATA_ROOT, TRAIN_CSV, tokenizer)
val_dataset = AVIDataset(DATA_ROOT, VAL_CSV, tokenizer)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=8,
    collate_fn=train_collate_fn,
    prefetch_factor=2,
    persistent_workers=True,
    generator=g,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=8,
    collate_fn=train_collate_fn,
    prefetch_factor=2,
    persistent_workers=True,
)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

In [ ]:
# Optimizer + scheduler + training loop

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=INITIAL_LR,
    betas=(0.9, 0.98),
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.2, patience=3, min_lr=2e-5,
)

scaler = torch.amp.GradScaler('cuda')

best_val_loss = float('inf')
epochs_no_improve = 0
epoch = 0

while True:
    print(f"\nEpoch {epoch}")

    optimizer.zero_grad()
    model.train()

    epoch_loss = 0.0
    current_accumulation_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch} [Train]")

    for i, batch in enumerate(pbar):
        videos, full_targets = batch
        if videos is None:
            continue

        videos = videos.to(device)
        full_targets = full_targets.to(device)

        dec_in = full_targets[:, :-1]
        target = full_targets[:, 1:]
        L = dec_in.size(1)
        tgt_mask = (subsequent_mask(L).to(device) & (dec_in != PAD_ID).unsqueeze(-2))

        with torch.amp.autocast('cuda'):
            with torch.no_grad():
                vid_emb = visual_encoder(videos)
            enc_B, enc_T, enc_D = vid_emb.size()
            src_mask = torch.ones((enc_B, 1, enc_T), device=device).bool()

            mem, _ = model.encode(vid_emb, src_mask)
            logits = model.decode(mem, src_mask, dec_in, tgt_mask)

            flat_targets = target.reshape(-1)
            loss = criterion(logits, flat_targets)
            loss = loss / ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        current_step_loss = loss.item() * ACCUMULATION_STEPS
        epoch_loss += current_step_loss
        current_accumulation_loss += current_step_loss

        if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

            pbar.set_postfix({'loss': current_accumulation_loss / ACCUMULATION_STEPS})
            current_accumulation_loss = 0.0

    train_loss = epoch_loss / max(len(train_loader), 1)

    # Validation
    val_loss = validate(visual_encoder, model, val_loader, criterion)

    print(f"\nEpoch {epoch} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"  LR: {optimizer.param_groups[0]['lr']:.2e}")

    # Save current
    torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "last_lm.pt"))

    # Save best
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_model.pt"))
        print(f"  -> New best ({best_val_loss:.4f})")
    else:
        epochs_no_improve += 1
        print(f"  No improvement ({epochs_no_improve}/{PATIENCE_STOP})")
        if epochs_no_improve >= PATIENCE_STOP:
            print(f"\nEarly stopping after {epoch + 1} epochs.")
            break

    scheduler.step(val_loss)
    epoch += 1

print(f"\nBest val loss: {best_val_loss:.4f}")
print(f"Best checkpoint: {os.path.join(OUTPUT_DIR, 'best_model.pt')}")